VRP의 과정을 통해서, COST가 최저인 경로를 분석하였다.
COST는 기타 소요 시간(충전, 승하차 등), 수요, 그리고 거리 이 3가지 요소를 반영하였다.

다만, 한계점이 존재했다. 
1. outgoing.csv를 통해서 시간대 별 인기 노선을 발견하였으나, 시간대 별 노선이었으므로 1시간만에 17개의 지역을 돌기는 매우 어려웠다.
2. 또한, HUB에서만 충전하도록 설정을 하였는데 배터리가 충분하였음에도 무조건 HUB만 돌면 충전이 되게끔 설정이 되었음
3. distance cost와 time cost를 하나로 통일시키는 작업을 하지 못하였음
4. ortools는 노선에 대한 fleet size를 하기에는 부적절한 툴이다. -> 교수님께서 조언해주신 대로 node couple를 여러개로 해서 진행을 해보았으나, 생성된 node couple은 모두 독립적이므로 fleet size 측정에 큰 오차가 발생할 것이다.

대안
1. 이번에는 Time Window의 개념을 도입.
-> 시간 단위 별로 수요 파악을 계속 진행하게 되므로, 위의 1의 한계점을 해결할 수 있다.(시간별로 이동하는 결과를 보여줄 수 있으므로)
-> 또한, 지금 vehicle을 1개로 하여 전체적인 수요 노선을 파악하고 있지만 추후 vehicle의 개수를 2개 이상으로 진행하게 된다면 한 node에 대해서 vehicle이 2개가 되는 불상사가 발생할 수 있다.
따라서, 이러한 문제점을 탈피하기 위해서 Time Window 개념을 도입해 Separation을 보장해보려고 한다.
이때, ortools의 경우에는 노드를 1개 밖에 경유할 수 밖에 없는 문제점이 존재하므로 이 문제를 해결하기 위해 NAME 만 다르고 모든 게 같은 NODE 여러 개(1, 11, 111, ....)를 생성해, if문을 이용해 vehicle 들이 Node list 중 2 군데 이상 위치한다면, collision으로 나오게 한다. (아마, if 문과 list를 활용하면 될 것 같음. 그렇게 len(list) => 2를 하면 되지 않을까 싶음)
-> 또한, 방문한 지역에 대해서 중복된 Node는 재방문 금지하도록 설정

2. UAV에 배터리 용량을 대입
-> 구체적인 mah나 용량이 아닌 [0,100]으로 설정해서 대략 160Km == 100%로 가정 (배터리 용량을 Capacity라 칭함)
-> 따라서, if Capacity < 30 -> depot(HUB) 이외의 node들의 penalty는 1000000으로 설정해 반드시 현 상황에서는 HUB를 방문하도록 함.
-> 또한, 충전시간도 100%를 한다고 가정
-> 이때, joby aviation에 따르면 20mile을 비행하기 위해서는 12mins의 충전시간을 요구한다고 기사에 나옴.
따라서, 1%를 충전하기 위해서는 약 36초가 걸린다고 가정할 수 있으므로 이 요소를 반영해 100% 완충할 때까지 진행.
-> 당연히, 충전시간 대에는 이동이 불가.(capacity가 1이라면 당연히 타 UAV 접근 불가. But, HUB의 Capacity를 얼마로 하냐에 따라 결과는 달라질 듯)

3. Fleet size는 다른 툴을 활용
-> ortools와는 호환성이 너무 안좋음

4. time과 distance에 대해서는 objectives value를 이용해 최대한 통일성 있게 진행.

추구하는 방향
1. 우선, 버스 노선과 같은 방식으로 진행해보려고 한다.
-> 택시와 같은 방식으로 진행을 하게 된다면, UAV 운항이 종료되었을 때 인기가 있는 지역들에 대해서 Capacity가 초과되는 우려사항이 존재한다
-> 만일, 이 방식을 채택하게 된다면 굳이 배터리 요소는 반영하지 않아도 됨. VRP 과정을 진행했을 때, 거의 어지간한 지역을 다 비행할 수 있는 것을 확인할 수 있었다. 따라서, 경영학 적인 접근을 한 번 해보면 어떨까(타당성 조사 등)
-> But, UAV가 모든 node를 경유하게 된다면 거리가 160Km를 초과하게 되어 반드시 1회 충전은 필연적이다. 따라서, 경유하는 node를 늘리게 된다면 충전은 필연이게 됨.
